## Notes de lecture

Ce notebook suit 4 étapes :

1. définir les points à visiter autour du dépôt de Rodez ;
2. appeler OSRM pour obtenir les matrices de distance et de durée ;
3. résoudre le problème de tournées avec OR-Tools ;
4. afficher les routes sur une carte Folium et exporter le résultat en HTML.


In [19]:
# Packages nécessaires au calcul d'itinéraires et à l'affichage de la carte
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
import requests
import folium

# Liste des points à visiter.
# Le premier point correspond au dépôt, les autres sont les visites à répartir.
locations = [
    ("Depot", "Rodez", 2.5734, 44.3526),

    ("A", "Baraqueville", 2.4318, 44.2766),
    ("B", "Flavin", 2.6032, 44.2889),
    ("C", "Saint-Côme-d'Olt", 2.8140, 44.5150),
    ("D", "Estaing", 2.6710, 44.5540),
    ("E", "Conques", 2.3970, 44.5990),
    ("F", "Valady", 2.4270, 44.4550),
    ("G", "Nauviale", 2.4260, 44.5200),
    ("H", "Firmi", 2.3100, 44.5400),
    ("I", "Cransac", 2.2840, 44.5250),
    ("J", "Balsac", 2.4450, 44.4010),
    ("K", "Villefranche-de-Rouergue", 2.0370, 44.3510),
    ("L", "Espalion", 2.7570, 44.5210),
    ("M", "Bozouls", 2.7240, 44.4700),
    ("N", "Laguiole", 2.8460, 44.6840),
    ("O", "Sévérac-d'Aveyron", 3.0520, 44.3230),
    ("P", "Millau", 3.0810, 44.1000),
    ("Q", "Saint-Affrique", 2.8850, 43.9580),
    ("R", "Pont-de-Salars", 2.7280, 44.2820),
    ("S", "Salles-Curan", 2.7880, 44.1820),
    ("T", "Réquista", 2.5350, 44.0330),
    ("U", "Decazeville", 2.2510, 44.5600),
    ("V", "Aubin", 2.2430, 44.5270),
    ("W", "Marcillac-Vallon", 2.4650, 44.4750),
    ("X", "Laissac", 2.6830, 44.3830),
    ("Y", "Rieupeyroux", 2.2360, 44.3050),

    ("Z", "Figeac", 2.0340, 44.6080),
    ("AA", "Cahors", 1.4410, 44.4490),
    ("AB", "Gourdon", 1.3820, 44.7360),
    ("AC", "Gramat", 1.7220, 44.7770),
    ("AD", "Saint-Céré", 1.8920, 44.8570),
    ("AE", "Souillac", 1.4730, 44.8960),
    ("AF", "Lacapelle-Marival", 1.9240, 44.7280),
    ("AG", "Limogne-en-Quercy", 1.7710, 44.3960),
    ("AH", "Puy-l'Évêque", 1.1370, 44.5040),
    ("AI", "Castelnau-Montratier", 1.3550, 44.2690),

        ("AJ", "Albi", 2.1480, 43.9280),
    ("AK", "Gaillac", 1.8970, 43.9020),
    ("AL", "Carmaux", 2.1580, 44.0490),
    ("AM", "Cordes-sur-Ciel", 1.9540, 44.0640),
    ("AN", "Graulhet", 1.9890, 43.7650),
    ("AO", "Lavaur", 1.8120, 43.6990),
    ("AP", "Castres", 2.2400, 43.6060),
    ("AQ", "Mazamet", 2.3720, 43.4920),
    ("AR", "Lisle-sur-Tarn", 1.8120, 43.8520),
    ("AS", "Rabastens", 1.7250, 43.8220),
]

# OSRM attend des coordonnées sous la forme lon,lat séparées par des points-virgules.
coordinates = ";".join(
    f"{lon},{lat}"
    for _, _, lon, lat in locations
)

# Appel à l'API table d'OSRM pour récupérer les distances et les durées entre tous les points.
url = (
    "http://127.0.0.1:5001/table/v1/driving/"
    + coordinates
    + "?annotations=distance,duration"
)

print(url)


# Les réponses OSRM sont en mètres et en secondes, on arrondit pour simplifier le travail de l'optimiseur.
response = requests.get(url)
response.raise_for_status()

data = response.json()

distance_matrix = [
    [round(value) for value in row]
    for row in data["distances"]
]

duration_matrix = [
    [round(value) for value in row]
    for row in data["durations"]
]

# La matrice de distance sert de coût principal pour OR-Tools.
print(distance_matrix)
print(duration_matrix)


# Résolution du VRP avec 4 véhicules et un dépôt unique.
def solve_vrp():
    print("debut")
    
    vehicle_count = 8
    depot_index = 0

    manager = pywrapcp.RoutingIndexManager(
        len(distance_matrix),
        vehicle_count,
        depot_index,
    )

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        return distance_matrix[from_node][to_node]

    
    def duration_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return duration_matrix[from_node][to_node]
    
    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    duration_callback_index = routing.RegisterTransitCallback(duration_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # DEBUT - si on enlève ce bloc, on se retrouve avec des véhocule qui font énormément de distance et d'autre presque aucune
    # en effet, sans ce bloc on minimize distance totale = distance véhicule 1 + distance véhicule 2 + ...
    # on évite avec ce bloc  qu’un véhicule ait une route beaucoup plus longue que les autres; 
    # OR-Tools essaie alors de minimiser le span, c’est-à-dire l’écart entre la route la plus courte et la plus longue, ou plus simplement le poids du véhicule qui fait le plus de distance.
    routing.AddDimension(
        transit_callback_index,
        0,
        2000000,  # max distance per vehicle in meters
        True,
        "Distance",
    )
    
    distance_dimension = routing.GetDimensionOrDie("Distance")
    
    # Important: this makes OR-Tools balance routes
    #With `setGlobalSpanCostCoefficient(100)` OR-Tools adds another objective: *minimize the longest route*
    #Solution A
    #- Truck 1 = 12 km
    #- Truck 2 = 95 km
    #- Longest = 95 km
    #- => Penalty: 95 × 100 = 9500
    
    #Solution B
    #- Truck 1 = 55 km
    #- Truck 2 = 60 km
    #- Longest = 60 km
    #- => Penaliyt = 60 × 100 = 6000
    distance_dimension.SetGlobalSpanCostCoefficient(100)
    # FIN

    routing.AddDimension(
        duration_callback_index,
        0,
        5 * 3600,  # max 8 hours per vehicle
        True,
        "Duration",
    )

    duration_dimension = routing.GetDimensionOrDie("Duration")
    

    for vehicle_id in range(vehicle_count):
        routing.solver().Add(
            routing.NextVar(routing.Start(vehicle_id)) != routing.End(vehicle_id)
        )
    
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = 5

    solution = routing.SolveWithParameters(search_parameters)

    if solution is None:
        print("No solution found")
        return

    routes = []
    
    for vehicle_id in range(vehicle_count):
        index = routing.Start(vehicle_id)
    
        route = []
        route_distance = 0
    
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route.append(node)
    
            previous_index = index
            index = solution.Value(routing.NextVar(index))
    
            route_distance += routing.GetArcCostForVehicle(
                previous_index,
                index,
                vehicle_id,
            )
    
        route.append(manager.IndexToNode(index))
    
        print(f"Vehicle {vehicle_id}:")
        
        route_labels = [
            f"{locations[node][0]} ({locations[node][1]})"
            for node in route
        ]
        
        print(" -> ".join(route_labels))
        
        print(f"Distance: {route_distance / 1000:.2f} km")
        print()
        routes.append({
            "vehicle_id": vehicle_id,
            "nodes": route,
            "distance": route_distance,
        })
        
    return routes


# Récupère la géométrie exacte d'une tournée auprès d'OSRM pour l'afficher sur la carte.
def get_osrm_geometry(route_nodes):
    coordinates = ";".join(
        f"{locations[node][2]},{locations[node][3]}"
        for node in route_nodes
    )

    url = (
        "http://127.0.0.1:5001/route/v1/driving/"
        + coordinates
        + "?overview=full&geometries=geojson"
    )

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    # OSRM returns [lon, lat], Folium needs [lat, lon]
    return [
        [lat, lon]
        for lon, lat in data["routes"][0]["geometry"]["coordinates"]
    ]


# Construit la carte Folium, place les points et trace chaque tournée avec une couleur dédiée.
def display_routes_on_map(routes):
    depot_lat = locations[0][3]
    depot_lon = locations[0][2]

    m = folium.Map(
        location=[depot_lat, depot_lon],
        zoom_start=10,
        tiles="CartoDB positron",
    )

    colors = ["blue", "red", "green", "purple", "orange", "darkred"]

    for name, city, lon, lat in locations:
        folium.Marker(
            location=[lat, lon],
            popup=f"{name} - {city}",
            tooltip=f"{name} ({city})",
            icon=folium.Icon(
                color="black" if name == "Depot" else "cadetblue",
                icon="home" if name == "Depot" else "info-sign",
            ),
        ).add_to(m)

    for route in routes:
        vehicle_id = route["vehicle_id"]
        route_nodes = route["nodes"]
        color = colors[vehicle_id % len(colors)]

        geometry = get_osrm_geometry(route_nodes)

        folium.PolyLine(
            geometry,
            color=color,
            weight=5,
            opacity=0.8,
            tooltip=f"Vehicle {vehicle_id} - {route['distance'] / 1000:.2f} km",
        ).add_to(m)

    return m


# Lance l'optimisation puis exporte la carte finale dans vrp_routes.html.
if __name__ == "__main__":
    routes = solve_vrp()
    print("display")
    m = display_routes_on_map(routes)
    m
    m.save("vrp_routes.html")

http://127.0.0.1:5001/table/v1/driving/2.5734,44.3526;2.4318,44.2766;2.6032,44.2889;2.814,44.515;2.671,44.554;2.397,44.599;2.427,44.455;2.426,44.52;2.31,44.54;2.284,44.525;2.445,44.401;2.037,44.351;2.757,44.521;2.724,44.47;2.846,44.684;3.052,44.323;3.081,44.1;2.885,43.958;2.728,44.282;2.788,44.182;2.535,44.033;2.251,44.56;2.243,44.527;2.465,44.475;2.683,44.383;2.236,44.305;2.034,44.608;1.441,44.449;1.382,44.736;1.722,44.777;1.892,44.857;1.473,44.896;1.924,44.728;1.771,44.396;1.137,44.504;1.355,44.269;2.148,43.928;1.897,43.902;2.158,44.049;1.954,44.064;1.989,43.765;1.812,43.699;2.24,43.606;2.372,43.492;1.812,43.852;1.725,43.822?annotations=distance,duration
[[0, 20886, 9764, 35168, 36883, 37750, 19874, 26568, 34386, 36275, 16866, 59999, 30930, 22252, 54270, 48091, 69658, 79320, 23327, 38108, 47782, 39251, 39864, 19258, 14366, 37610, 65696, 112320, 126788, 105860, 110751, 136447, 90130, 77610, 146341, 120800, 71941, 100095, 61732, 84696, 112823, 129182, 114721, 134414, 106657, 116501], [